## Next Steps

This notebook covered evaluation basics in trainlib. For more advanced topics, see:

- **06_advanced.ipynb**: Custom metrics, data formats, and training callbacks
- **Documentation**: Full API reference for evaluation at [trainlib.dev/docs/eval](https://trainlib.dev/docs/eval)
- **Benchmark catalog**: Available benchmarks at [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)

Key takeaways:
- Use `trainlib.evaluate()` for quick model evaluation
- Register custom metrics with `@register_metric` for domain-specific needs
- Leverage `benchmark_evaluate()` for standardized comparisons
- Configure `EvalConfig` to monitor training progress

In [ ]:
from trainlib.config.schema import TrainConfig, ModelConfig, DataConfig, TrainerConfig, EvalConfig

config = TrainConfig(
    recipe="finetune",
    method="lora",
    model=ModelConfig(name="meta-llama/Llama-3.1-8B"),
    data=DataConfig(path="data/train.jsonl", format="alpaca", eval_split=0.1),
    trainer=TrainerConfig(batch_size=4, num_epochs=3),
    eval=EvalConfig(
        every_n_steps=500,
        metrics=["loss", "perplexity"],
    ),
)

state = trainlib.finetune(config=config)

# Evaluation results are logged to TensorBoard and returned in the training state
print(f"Final eval loss: {state.eval_metrics['loss']:.4f}")
print(f"Final eval perplexity: {state.eval_metrics['perplexity']:.2f}")

## Evaluation During Training

Configure automatic evaluation during training to monitor progress. The `EvalConfig` specifies evaluation frequency and metrics:

- `every_n_steps`: How often to run evaluation (in training steps)
- `metrics`: Which metrics to compute
- trainlib automatically uses the validation split specified in `DataConfig.eval_split`

In [ ]:
from trainlib.eval import benchmark_evaluate

benchmarks = ["mmlu", "hellaswag", "gsm8k"]

results_base = benchmark_evaluate(model="meta-llama/Llama-3.1-8B", benchmarks=benchmarks)
results_finetuned = benchmark_evaluate(model="output/my-finetuned-model", benchmarks=benchmarks)

# Compare results
print(f"{'Benchmark':<15} {'Base':<10} {'Fine-tuned':<10} {'Delta':<10}")
print("-" * 45)
for bench in benchmarks:
    base_acc = results_base.get(bench, {}).get("acc,none", 0)
    ft_acc = results_finetuned.get(bench, {}).get("acc,none", 0)
    delta = ft_acc - base_acc
    print(f"{bench:<15} {base_acc:<10.4f} {ft_acc:<10.4f} {delta:+.4f}")

## Model Comparison

Compare a base model against a fine-tuned version to measure improvement. This is especially useful for validating that fine-tuning improved performance on your target benchmarks:

In [ ]:
results = benchmark_evaluate(
    model="output/my-model",
    benchmarks=["mmlu"],
    num_fewshot=5,
)

print(f"MMLU (5-shot):")
for metric_name, value in results["mmlu"].items():
    print(f"  {metric_name}: {value}")

## Few-Shot Evaluation

Many benchmarks support few-shot evaluation, where the model is shown a few examples before being tested. Configure this with the `num_fewshot` parameter:

In [ ]:
from trainlib.eval import benchmark_evaluate

# Run standard benchmarks
results = benchmark_evaluate(
    model="output/my-model",
    benchmarks=["mmlu", "hellaswag", "gsm8k"],
)

for benchmark, metrics in results.items():
    print(f"\n{benchmark}:")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value}")

## Benchmark Evaluation

trainlib integrates with [lm-eval-harness](https://github.com/EleutherAI/lm-evaluation-harness) for standardized benchmarks. This requires installing the evaluation extras:

```bash
pip install trainlib[eval]
```

Run standard benchmarks like MMLU, HellaSwag, and GSM8K:

In [ ]:
from trainlib.eval import register_metric

@register_metric("exact_match")
def exact_match(predictions, references, *args, **kwargs):
    """Fraction of predictions that exactly match references."""
    if not predictions:
        return 0.0
    return sum(p == r for p, r in zip(predictions, references)) / len(predictions)

@register_metric("f1_score")
def f1_score(predictions, references, *args, **kwargs):
    """Token-level F1 score."""
    if not predictions:
        return 0.0
    pred_set = set(predictions)
    ref_set = set(references)
    if not pred_set or not ref_set:
        return 0.0
    precision = len(pred_set & ref_set) / len(pred_set)
    recall = len(pred_set & ref_set) / len(ref_set)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

# Use custom metrics
results = trainlib.evaluate(
    model="output/my-model",
    dataset=eval_data,
    metrics=["loss", "exact_match", "f1_score"],
)

print(f"Loss: {results['loss']:.4f}")
print(f"Exact Match: {results['exact_match']:.2%}")
print(f"F1 Score: {results['f1_score']:.4f}")

## Custom Metrics

Define domain-specific metrics using the `@register_metric` decorator. Custom metrics receive predictions and references and return a scalar value.

This example shows exact-match and F1 metrics for text generation tasks:

In [ ]:
results = trainlib.evaluate(
    model="output/my-model",
    dataset=eval_data,
    metrics=["loss", "perplexity", "token_accuracy"],
)

print(f"Loss: {results['loss']:.4f}")
print(f"Perplexity: {results['perplexity']:.2f}")
print(f"Token Accuracy: {results['token_accuracy']:.2%}")

## Built-in Metrics

trainlib includes three core metrics out of the box:

- **loss**: Average cross-entropy loss over the dataset
- **perplexity**: Exponential of loss, measures model uncertainty (lower is better)
- **token_accuracy**: Fraction of tokens predicted correctly

You can compute all metrics in a single call:

In [ ]:
import trainlib

results = trainlib.evaluate(
    model="output/my-model",
    dataset=[
        {"input_ids": [1, 2, 3], "labels": [1, 2, 3]},
        {"input_ids": [4, 5, 6], "labels": [4, 5, 6]},
    ],
    metrics=["loss", "perplexity"],
)

print(f"Loss: {results['loss']:.4f}")
print(f"Perplexity: {results['perplexity']:.4f}")

## Basic Evaluation

Evaluate a model on a dataset with built-in metrics. The `trainlib.evaluate()` function takes a model path or checkpoint, a dataset (list of dictionaries with `input_ids` and `labels`), and a list of metrics to compute.

# Evaluation & Benchmarks

trainlib provides a comprehensive evaluation system for assessing model performance:

- **Built-in metrics**: loss, perplexity, token accuracy
- **Custom metric registration**: define domain-specific metrics with a simple decorator
- **Benchmark integration**: lm-eval-harness support for standard benchmarks (MMLU, HellaSwag, GSM8K, etc.)
- **Model comparison**: programmatic tools for comparing base vs fine-tuned models
- **Training-time evaluation**: configure automated eval during training runs

This notebook demonstrates evaluation workflows. Code cells show exact API usage but won't execute without GPU/models.